# LeetCode 998 — Maximum Binary Tree II bằng Python

Link đề: https://leetcode.com/problems/maximum-binary-tree-ii/ · Độ khó: **Medium** · Chủ đề: Tree (cây), Binary Tree (cây nhị phân)

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def insertIntoMaxTree(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        ...
```

Ghi chú: **cây cực đại** (maximum tree) là cây mà mỗi nút lớn hơn mọi giá trị trong cây con của nó. Cây được dựng từ một mảng `a` bằng quy tắc `Construct(a)`: lấy phần tử lớn nhất làm gốc, nửa **trái** của mảng dựng thành cây con trái, nửa **phải** dựng thành cây con phải. Ta chỉ được cho cây `root = Construct(a)`, **không** được cho `a`. Hãy trả về `Construct(b)` với `b = a + [val]` (thêm `val` vào **cuối** `a`). Ý then chốt: giá trị mới nằm **cuối** mảng nên nó luôn thuộc nhánh **phải ngoài cùng** của cây. Bài này **tự dựng `TreeNode`** và hàm dựng/xuất cây theo danh sách kiểu LeetCode để kiểm thử. Vì cây `Construct(b)` là **duy nhất** nên ta so sánh **đúng cấu trúc** (không phải bài nhiều đáp án).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu quy tắc `Construct` và quan hệ giữa **cây cực đại** với **duyệt thứ tự giữa** (inorder = mảng `a`).
2. Nắm các công cụ Python cần thiết: lớp `TreeNode`, hàm dựng cây từ danh sách theo mức, đệ quy trên cây trả về gốc mới, duyệt vòng lặp theo nhánh phải, và ngăn xếp đơn điệu.
3. Hiểu nguyên lý: thêm phần tử vào **cuối** mảng chỉ ảnh hưởng **nhánh phải ngoài cùng**; đi xuống theo nhánh phải cho tới khi gặp nút nhỏ hơn `val`, rồi **chèn nút mới ở đó** và cho phần cây cũ làm **con trái** của nút mới.
4. Đi qua 4 cách: dựng lại từ đầu bằng định nghĩa (khôi phục `a` bằng inorder, thêm `val`, `Construct`), đệ quy theo nhánh phải (cách nộp chính), vòng lặp theo nhánh phải, và dựng lại bằng ngăn xếp đơn điệu trong `O(n)`.
5. Tránh các lỗi kinh điển: **mất** phần cây cũ (không gắn làm con trái), đi xuống **nhánh trái** thay vì phải, gắn `val` làm con phải của nút cuối cùng bất kể giá trị, ghi đè `root.val`, và quên trả về **gốc mới** khi `val` lớn nhất.

## 2. Đề bài & ví dụ

Một **cây cực đại** là cây mà mọi nút có giá trị **lớn hơn** mọi giá trị khác trong cây con của nó.

Cho `root` của một cây cực đại và số nguyên `val`.

Như bài trước (654), cây được dựng từ danh sách `a` (`root = Construct(a)`) theo quy tắc đệ quy `Construct(a)`:

- nếu `a` rỗng, trả về `null`;
- ngược lại, gọi `a[i]` là phần tử **lớn nhất** của `a`; tạo nút gốc có giá trị `a[i]`;
- con **trái** của gốc là `Construct([a[0], ..., a[i - 1]])`;
- con **phải** của gốc là `Construct([a[i + 1], ..., a[a.length - 1]])`;
- trả về gốc.

Lưu ý ta **không** được cho `a`, chỉ cho `root = Construct(a)`. Giả sử `b` là bản sao của `a` với `val` được thêm vào **cuối**; bảo đảm `b` có các giá trị **khác nhau**. Trả về `Construct(b)`.

**Ví dụ 1:** `root = [4,1,3,null,null,2]`, `val = 5` -> `[5,4,null,1,3,null,null,2]` (`a = [1,4,2,3]`, `b = [1,4,2,3,5]`).

**Ví dụ 2:** `root = [5,2,4,null,1]`, `val = 3` -> `[5,2,4,null,1,null,3]` (`a = [2,1,5,4]`, `b = [2,1,5,4,3]`).

**Ví dụ 3:** `root = [5,2,3,null,1]`, `val = 4` -> `[5,2,4,null,1,3]` (`a = [2,1,5,3]`, `b = [2,1,5,3,4]`).

**Ràng buộc:**

- Số nút trong cây thuộc `[1, 100]`.
- `1 <= Node.val <= 100`.
- Mọi giá trị trong cây **khác nhau**.
- `1 <= val <= 100`.

In [1]:
from typing import List, Optional
from collections import deque
import random, time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(vals: List[Optional[int]]) -> Optional[TreeNode]:
    """Dựng cây từ danh sách theo mức kiểu LeetCode (None là nút trống; chỉ nút thật có hai phần tử con trong danh sách)."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = deque([root]), 1
    while queue and i < len(vals):
        node = queue.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root

def tree_to_list(root: Optional[TreeNode]) -> List[Optional[int]]:
    """Xuất cây theo mức kiểu LeetCode (bỏ None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def inorder(root: Optional[TreeNode]) -> List[int]:
    return [] if root is None else inorder(root.left) + [root.val] + inorder(root.right)

def is_max_tree(root: Optional[TreeNode]) -> bool:
    """Mỗi nút lớn hơn mọi giá trị trong cây con của nó."""
    def mx(node):
        if node is None:
            return None
        m = node.val
        for ch in (node.left, node.right):
            if ch is not None:
                cm = mx(ch)
                if cm is None or cm >= node.val:
                    raise ValueError
                m = max(m, cm)
        return m
    try:
        mx(root)
        return True
    except ValueError:
        return False

examples = [
    ([4, 1, 3, None, None, 2], 5, [5, 4, None, 1, 3, None, None, 2]),
    ([5, 2, 4, None, 1], 3, [5, 2, 4, None, 1, None, 3]),
    ([5, 2, 3, None, 1], 4, [5, 2, 4, None, 1, 3]),
]
for vals, val, expected in examples:
    t = build_tree(vals)
    print(f"root={vals} val={val}: inorder (= a) = {inorder(t)}, là cây cực đại: {is_max_tree(t)}, kỳ vọng {expected}")
    assert tree_to_list(t) == vals and is_max_tree(t)

root=[4, 1, 3, None, None, 2] val=5: inorder (= a) = [1, 4, 2, 3], là cây cực đại: True, kỳ vọng [5, 4, None, 1, 3, None, None, 2]
root=[5, 2, 4, None, 1] val=3: inorder (= a) = [2, 1, 5, 4], là cây cực đại: True, kỳ vọng [5, 2, 4, None, 1, None, 3]
root=[5, 2, 3, None, 1] val=4: inorder (= a) = [2, 1, 5, 3], là cây cực đại: True, kỳ vọng [5, 2, 4, None, 1, 3]


## 3. Kiến thức nền

### 3.1. `Construct` và duyệt thứ tự giữa

Trong `Construct(a)`, phần tử lớn nhất `a[i]` là gốc, nửa trái `a[0..i-1]` ở cây con trái, nửa phải ở cây con phải. Vì vậy **duyệt thứ tự giữa** (trái, gốc, phải) của cây trả lại đúng mảng `a`. Ngược lại, từ `a` ta dựng được cây duy nhất. Ta viết `construct(a)` đúng theo định nghĩa để làm **oracle**.

In [2]:
def construct(a: List[int]) -> Optional[TreeNode]:
    """Construct(a) đúng theo định nghĩa của đề (a có các phần tử khác nhau)."""
    if not a:
        return None
    i = a.index(max(a))
    return TreeNode(a[i], construct(a[:i]), construct(a[i + 1:]))

t = construct([1, 4, 2, 3])
print("Construct([1,4,2,3]) =", tree_to_list(t), "| inorder =", inorder(t))
assert tree_to_list(t) == [4, 1, 3, None, None, 2] and inorder(t) == [1, 4, 2, 3]

Construct([1,4,2,3]) = [4, 1, 3, None, None, 2] | inorder = [1, 4, 2, 3]


### 3.2. Nhánh phải ngoài cùng (right spine)

Chuỗi các nút đi từ gốc rồi luôn rẽ **phải** gọi là nhánh phải ngoài cùng. Trong cây cực đại, giá trị **giảm dần** dọc theo nhánh này. Phần tử **cuối** của mảng `a` (phần tử ở xa nhất về bên phải) chính là nút **cuối** của nhánh phải ngoài cùng.

In [3]:
def right_spine(root):
    out = []
    while root is not None:
        out.append(root.val)
        root = root.right
    return out

t = build_tree([5, 2, 4, None, 1])
print("nhánh phải ngoài cùng của [5,2,4,null,1]:", right_spine(t), "| a =", inorder(t), "(phần tử cuối của a là", inorder(t)[-1], ")")
assert right_spine(t)[-1] == inorder(t)[-1]

nhánh phải ngoài cùng của [5,2,4,null,1]: [5, 4] | a = [2, 1, 5, 4] (phần tử cuối của a là 4 )


### 3.3. Đệ quy trên cây trả về gốc mới

Hàm đệ quy `f(node)` có thể **trả về gốc của cây con đã sửa**; cha gán lại `parent.right = f(parent.right)`. Mẫu này tránh phải theo dõi nút cha thủ công.

In [4]:
def demo_replace_right(node, new_val):
    if node is None:
        return TreeNode(new_val)
    node.right = demo_replace_right(node.right, new_val)
    return node

t = build_tree([5, 2])
t = demo_replace_right(t, 9)
print(tree_to_list(t))
assert tree_to_list(t) == [5, 2, 9]

[5, 2, 9]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 3: `root = [5,2,3,null,1]` (`a = [2,1,5,3]`), `val = 4`, `b = [2,1,5,3,4]`.

- Nhánh phải ngoài cùng của cây: `5 -> 3`. Phần tử mới `4` đứng **cuối** của `b`.
- `4 < 5` nên gốc vẫn là `5` (vì `5` vẫn là lớn nhất); `4` thuộc nửa **phải** của `5`: ta xét cây con phải `[3]`.
- `4 > 3`: tại đây `4` lớn hơn mọi phần tử của cây con `[3]` nên `4` trở thành **gốc** của đoạn này, còn cây con cũ `3` trở thành **con trái** của `4`.
- Kết quả: `5` có con phải là `4` (con trái của `4` là `3`): cây `[5,2,4,null,1,3]`.

Ví dụ 1: `val = 5` lớn hơn gốc `4`, nên `5` là gốc mới và toàn bộ cây cũ là **con trái** của `5`: `[5,4,null,1,3,null,null,2]`.

Quy tắc: đi xuống nhánh phải cho đến khi gặp vị trí rỗng hoặc nút có giá trị **nhỏ hơn** `val`; chèn nút mới tại đó, và đặt cây con vừa bỏ qua làm **con trái** của nút mới.

## 5. Cách 1 — Brute Force: dựng lại từ đầu theo định nghĩa

**Ý tưởng.** Khôi phục `a` bằng **inorder**, tạo `b = a + [val]`, rồi gọi `construct(b)`. Đây là định nghĩa của đề: không cần hiểu cấu trúc nhánh phải. Mỗi lần `construct` tìm phần tử lớn nhất bằng `max` và `index` rồi cắt mảng: `O(n^2)` tồi nhất (khi mảng đã sắp xếp), nhưng với `n <= 100` vẫn chạy nhanh. Đây cũng là **oracle độc lập**.

In [5]:
def insert_rebuild(root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
    a = inorder(root)
    return construct(a + [val])

for vals, val, expected in examples:
    got = tree_to_list(insert_rebuild(build_tree(vals), val))
    print(f"root={vals} val={val}: dựng lại = {got}", "OK" if got == expected else "SAI")
    assert got == expected

root=[4, 1, 3, None, None, 2] val=5: dựng lại = [5, 4, None, 1, 3, None, None, 2] OK
root=[5, 2, 4, None, 1] val=3: dựng lại = [5, 2, 4, None, 1, None, 3] OK
root=[5, 2, 3, None, 1] val=4: dựng lại = [5, 2, 4, None, 1, 3] OK


**Độ phức tạp và nút thắt.** Inorder `O(n)` cộng `construct` tồi nhất `O(n^2)` (cắt mảng và tìm max ở mỗi mức): với `n = 100` còn ổn, nhưng cách làm **bỏ phí** thông tin cấu trúc đã có: toàn bộ cây con trái, và phần lớn nhánh phải, **không đổi**. Đo thử với mảng tăng dần (tồi nhất):

In [6]:
for n in (50, 100, 400, 800):
    a = list(range(1, n + 1))
    root = construct(a)
    t0 = time.perf_counter()
    for _ in range(5):
        insert_rebuild(root, n + 1)
    print(f"n={n:4}: {(time.perf_counter() - t0) / 5 * 1000:8.2f} ms mỗi lần dựng lại")

n=  50:     0.09 ms mỗi lần dựng lại
n= 100:     0.28 ms mỗi lần dựng lại
n= 400:     3.42 ms mỗi lần dựng lại
n= 800:     8.64 ms mỗi lần dựng lại


Nút thắt: ta dựng lại **toàn bộ** cây dù chỉ nhánh phải ngoài cùng bị ảnh hưởng. Ta chỉ cần sửa nhánh phải.

## 6. Ý tưởng thuật toán / bước đột phá

**`val` nằm cuối mảng nên chỉ ảnh hưởng nhánh phải ngoài cùng.**

Xét `root = Construct(a)` và `b = a + [val]`:

- Nếu `val > root.val` thì `val` là phần tử lớn nhất của `b`: nó là gốc mới, nửa trái của `b` là toàn bộ `a`, nửa phải rỗng. Vậy `Construct(b) = Node(val, left=root, right=None)`.
- Nếu `val < root.val` thì `root.val` vẫn là lớn nhất của `b`, vị trí của nó trong `b` không đổi, nửa trái không đổi (không chứa `val`); `val` nằm trong nửa **phải**: `root.right = Construct(nửa phải + [val])`, tức **cùng bài toán** trên cây con phải.
- Nếu cây rỗng: `Construct([val]) = Node(val)`.

Đó là đệ quy `insert(root, val)`; mỗi bước đi xuống một nút trên nhánh phải: `O(h)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Đệ quy: `val > root.val` thì bọc cây cũ làm con trái, ngược lại đệ quy sang con phải (nộp) |
| 3 | Vòng lặp: đi xuống nhánh phải giữ nút cha, chèn nút mới rồi nối lại |
| 4 | Khôi phục `a` bằng inorder, thêm `val`, dựng lại bằng ngăn xếp đơn điệu `O(n)` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy theo nhánh phải (cách nộp chính)

**Giải thích từng đoạn.**

- `if root is None or val > root.val`: nút mới `TreeNode(val)` thay vào vị trí này, và cây con cũ `root` trở thành **con trái** của nút mới (`left=root`). Nếu `root is None` thì `left=None`.
- Ngược lại `root.right = insert(root.right, val)` rồi `return root`.

In [7]:
def insert_recursive(root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
    if root is None or val > root.val:
        return TreeNode(val, left=root)
    root.right = insert_recursive(root.right, val)
    return root

for vals, val, expected in examples:
    got = tree_to_list(insert_recursive(build_tree(vals), val))
    print(f"root={vals} val={val}: đệ quy = {got}", "OK" if got == expected else "SAI")
    assert got == expected

root=[4, 1, 3, None, None, 2] val=5: đệ quy = [5, 4, None, 1, 3, None, None, 2] OK
root=[5, 2, 4, None, 1] val=3: đệ quy = [5, 2, 4, None, 1, None, 3] OK
root=[5, 2, 3, None, 1] val=4: đệ quy = [5, 2, 4, None, 1, 3] OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def insertIntoMaxTree(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        if root is None or val > root.val:
            return TreeNode(val, left=root)          # nút mới: cây cũ thành con trái
        root.right = self.insertIntoMaxTree(root.right, val)
        return root

sol_recursive = Solution().insertIntoMaxTree
for vals, val, expected in examples:
    assert tree_to_list(sol_recursive(build_tree(vals), val)) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(h)` với `h` là chiều dài nhánh phải ngoài cùng (`<= n <= 100`); bộ nhớ `O(h)` cho ngăn xếp đệ quy.

---

### 7.2. Cách 3 — Vòng lặp theo nhánh phải

**Ý tưởng khác.** Đi xuống nhánh phải giữ hai con trỏ `parent` (nút trước) và `cur` (nút hiện tại) cho tới khi `cur` rỗng hoặc `cur.val < val`. Tạo nút mới `node = TreeNode(val, left=cur)`. Nếu `parent` là `None` (nghĩa là `val` lớn hơn gốc, hoặc cây rỗng) thì **nút mới là gốc mới**; ngược lại `parent.right = node`.

**Giải thích từng đoạn.**

- `while cur is not None and cur.val > val`: tiếp tục đi xuống nhánh phải khi nút hiện tại vẫn lớn hơn `val` (giá trị khác nhau nên không có hòa).
- Sau vòng lặp `cur` là nút đầu tiên nhỏ hơn `val` (hoặc rỗng): chèn nút mới ngay trước `cur`.

In [9]:
def insert_iterative(root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
    parent, cur = None, root
    while cur is not None and cur.val > val:
        parent, cur = cur, cur.right
    node = TreeNode(val, left=cur)
    if parent is None:
        return node                                   # val lớn nhất: nút mới là gốc
    parent.right = node
    return root

for vals, val, expected in examples:
    got = tree_to_list(insert_iterative(build_tree(vals), val))
    print(f"root={vals} val={val}: vòng lặp = {got}", "OK" if got == expected else "SAI")
    assert got == expected

root=[4, 1, 3, None, None, 2] val=5: vòng lặp = [5, 4, None, 1, 3, None, None, 2] OK
root=[5, 2, 4, None, 1] val=3: vòng lặp = [5, 2, 4, None, 1, None, 3] OK
root=[5, 2, 3, None, 1] val=4: vòng lặp = [5, 2, 4, None, 1, 3] OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def insertIntoMaxTree(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        parent, cur = None, root
        while cur is not None and cur.val > val:     # đi xuống nhánh phải ngoài cùng
            parent, cur = cur, cur.right
        node = TreeNode(val, left=cur)               # cây con vừa bỏ qua thành con trái của nút mới
        if parent is None:
            return node                              # val lớn nhất (hoặc cây rỗng): nút mới là gốc
        parent.right = node
        return root

sol_iterative = Solution().insertIntoMaxTree
for vals, val, expected in examples:
    assert tree_to_list(sol_iterative(build_tree(vals), val)) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(h)`, bộ nhớ `O(1)` thêm (không đệ quy).

---

### 7.3. Cách 4 — Khôi phục `a` rồi dựng lại bằng ngăn xếp đơn điệu

**Ý tưởng khác.** Ta vẫn khôi phục `b = inorder(root) + [val]`, nhưng thay `construct` chậm bằng thuật toán `O(n)` dùng **ngăn xếp giảm dần** (Cartesian tree): duyệt `b` từ trái sang phải; với mỗi `x`, **pop** mọi nút nhỏ hơn `x` khỏi ngăn xếp (nút pop cuối cùng trở thành **con trái** của `x`), rồi nếu ngăn xếp còn nút thì `x` là **con phải** của nút trên cùng; đẩy `x` vào ngăn xếp. Gốc là phần tử đáy ngăn xếp.

**Giải thích từng đoạn.**

- `last` là nút cuối cùng bị pop (lớn nhất trong các nút bị pop vì ngăn xếp giảm dần từ đáy lên đỉnh, nên pop theo thứ tự tăng dần).
- Gán `node.left = last` giữ nguyên cây con đã dựng bên trái.
- `stack[-1].right = node`: ghi đè con phải cũ của nút trên cùng (con phải cũ đã trở thành `node.left` hoặc nằm trong đó).

In [11]:
def construct_stack(a: List[int]) -> Optional[TreeNode]:
    stack: List[TreeNode] = []
    for x in a:
        node = TreeNode(x)
        last = None
        while stack and stack[-1].val < x:
            last = stack.pop()
        node.left = last
        if stack:
            stack[-1].right = node
        stack.append(node)
    return stack[0] if stack else None

def insert_stack(root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
    return construct_stack(inorder(root) + [val])

for vals, val, expected in examples:
    got = tree_to_list(insert_stack(build_tree(vals), val))
    print(f"root={vals} val={val}: ngăn xếp = {got}", "OK" if got == expected else "SAI")
    assert got == expected

root=[4, 1, 3, None, None, 2] val=5: ngăn xếp = [5, 4, None, 1, 3, None, None, 2] OK
root=[5, 2, 4, None, 1] val=3: ngăn xếp = [5, 2, 4, None, 1, None, 3] OK
root=[5, 2, 3, None, 1] val=4: ngăn xếp = [5, 2, 4, None, 1, 3] OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def insertIntoMaxTree(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        a = []

        def inorder_walk(node):
            if node is None:
                return
            inorder_walk(node.left)
            a.append(node.val)
            inorder_walk(node.right)

        inorder_walk(root)
        a.append(val)                                 # b = a + [val]
        stack = []
        for x in a:                                   # dựng cây cực đại bằng ngăn xếp giảm dần
            node = TreeNode(x)
            last = None
            while stack and stack[-1].val < x:
                last = stack.pop()
            node.left = last
            if stack:
                stack[-1].right = node
            stack.append(node)
        return stack[0]

sol_stack = Solution().insertIntoMaxTree
for vals, val, expected in examples:
    assert tree_to_list(sol_stack(build_tree(vals), val)) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n)` (inorder `O(n)`, dựng bằng ngăn xếp `O(n)`), bộ nhớ `O(n)`: chậm hơn Cách 2/3 (chỉ `O(h)`), nhưng cho thấy cách dựng cây cực đại `O(n)`.

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 3 (`root = [5,2,3,null,1]`, `val = 4`)

In từng lời gọi đệ quy: nút hiện tại, so sánh với `val`, và hành động.

In [13]:
def trace_insert(root, val, depth=0):
    pad = "  " * depth
    if root is None:
        print(f"{pad}insert(None, {val}) -> tạo nút {val} (không có cây con cũ)")
        return TreeNode(val)
    if val > root.val:
        print(f"{pad}insert(nút {root.val}, {val}): {val} > {root.val} -> nút mới {val}, cây con gốc {root.val} thành CON TRÁI")
        return TreeNode(val, left=root)
    print(f"{pad}insert(nút {root.val}, {val}): {val} < {root.val} -> đi xuống con PHẢI")
    root.right = trace_insert(root.right, val, depth + 1)
    return root

t = trace_insert(build_tree([5, 2, 3, None, 1]), 4)
print("kết quả:", tree_to_list(t))
assert tree_to_list(t) == [5, 2, 4, None, 1, 3]

insert(nút 5, 4): 4 < 5 -> đi xuống con PHẢI
  insert(nút 3, 4): 4 > 3 -> nút mới 4, cây con gốc 3 thành CON TRÁI
kết quả: [5, 2, 4, None, 1, 3]


### 8.2. Cách 3 cho ví dụ 1 (`val = 5` lớn hơn gốc)

Vòng lặp không đi xuống lần nào (`4 > 5` sai): `parent` là `None`, nút mới `5` là gốc mới và cây cũ là con trái.

In [14]:
def trace_iter(root, val):
    parent, cur = None, root
    step = 0
    while cur is not None and cur.val > val:
        print(f"bước {step}: cur={cur.val} > {val} -> xuống phải (parent={cur.val})")
        parent, cur = cur, cur.right
        step += 1
    print(f"dừng: cur={None if cur is None else cur.val}, parent={None if parent is None else parent.val}")
    node = TreeNode(val, left=cur)
    if parent is None:
        print("parent là None: nút mới là gốc mới")
        return node
    parent.right = node
    return root

res = trace_iter(build_tree([4, 1, 3, None, None, 2]), 5)
print("kết quả:", tree_to_list(res))
assert tree_to_list(res) == [5, 4, None, 1, 3, None, None, 2]
res = trace_iter(build_tree([5, 2, 4, None, 1]), 3)
print("kết quả:", tree_to_list(res))
assert tree_to_list(res) == [5, 2, 4, None, 1, None, 3]

dừng: cur=4, parent=None
parent là None: nút mới là gốc mới
kết quả: [5, 4, None, 1, 3, None, None, 2]
bước 0: cur=5 > 3 -> xuống phải (parent=5)
bước 1: cur=4 > 3 -> xuống phải (parent=4)
dừng: cur=None, parent=4
kết quả: [5, 2, 4, None, 1, None, 3]


### 8.3. Ngăn xếp đơn điệu cho `b = [2, 1, 5, 3, 4]`

In ngăn xếp (từ đáy đến đỉnh) sau mỗi phần tử.

In [15]:
def trace_stack(a):
    stack = []
    for x in a:
        node = TreeNode(x)
        last = None
        popped = []
        while stack and stack[-1].val < x:
            last = stack.pop()
            popped.append(last.val)
        node.left = last
        if stack:
            stack[-1].right = node
        stack.append(node)
        print(f"x={x}: pop {popped}, left={None if last is None else last.val}, ngăn xếp = {[s.val for s in stack]}")
    return stack[0]

t = trace_stack([2, 1, 5, 3, 4])
print("cây:", tree_to_list(t))
assert tree_to_list(t) == [5, 2, 4, None, 1, 3]

x=2: pop [], left=None, ngăn xếp = [2]
x=1: pop [], left=None, ngăn xếp = [2, 1]
x=5: pop [1, 2], left=2, ngăn xếp = [5]
x=3: pop [], left=None, ngăn xếp = [5, 3]
x=4: pop [3], left=3, ngăn xếp = [5, 4]
cây: [5, 2, 4, None, 1, 3]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (đệ quy theo nhánh phải): ngắn, đúng, `O(h)`.

```python
class Solution:
    def insertIntoMaxTree(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        if root is None or val > root.val:
            return TreeNode(val, left=root)          # nút mới: cây cũ thành con trái
        root.right = self.insertIntoMaxTree(root.right, val)
        return root
```

In [16]:
class Solution:
    def insertIntoMaxTree(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        if root is None or val > root.val:
            return TreeNode(val, left=root)          # nút mới: cây cũ thành con trái
        root.right = self.insertIntoMaxTree(root.right, val)
        return root

sol = Solution()
for vals, val, expected in examples:
    got = tree_to_list(sol.insertIntoMaxTree(build_tree(vals), val))
    print(f"root={vals} val={val}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

root=[4, 1, 3, None, None, 2] val=5: [5, 4, None, 1, 3, None, None, 2] OK
root=[5, 2, 4, None, 1] val=3: [5, 2, 4, None, 1, None, 3] OK
root=[5, 2, 3, None, 1] val=4: [5, 2, 4, None, 1, 3] OK


## 10. Vì sao đúng

Ký hiệu `a = inorder(root)`; bất biến: `root = Construct(a)`.

**Trường hợp `root` rỗng.** `a` rỗng, `b = [val]`, `Construct(b)` là một nút `val` không con: đúng với `TreeNode(val, left=None)`.

**Trường hợp `val > root.val`.** Vì `root.val` là phần tử lớn nhất của `a` và `val` lớn hơn nó, `val` là phần tử lớn nhất của `b`, nằm ở **cuối** `b`. Theo `Construct`, gốc là `val`, cây con trái là `Construct(b[0..-2]) = Construct(a) = root` và cây con phải là `Construct([])` rỗng. Đúng bằng `TreeNode(val, left=root)`.

**Trường hợp `val < root.val`.** Phần tử lớn nhất của `b` vẫn là `root.val` (với chỉ số `i` như trong `a`, vì `val` thêm vào cuối nhỏ hơn), nên `Construct(b)` có gốc `root.val`, cây con trái `Construct(a[0..i-1])` **không đổi** (chính là `root.left`), và cây con phải là `Construct(a[i+1..] + [val])`. Ở đó `a[i+1..]` có cây `Construct(a[i+1..]) = root.right`, nên cây con phải mới chính là kết quả của cùng bài toán trên `root.right` với `val`: đúng `root.right = insert(root.right, val)`. Quy nạp theo chiều cao của nhánh phải ngoài cùng.

**Cách 3** thực hiện cùng phép biến đổi bằng vòng lặp (chèn tại vị trí đầu tiên mà `cur` rỗng hoặc `cur.val < val`), và trả về nút mới khi `parent is None` (đúng trường hợp `val` lớn nhất).

**Cách 4** dựng lại `Construct(b)` bằng thuật toán ngăn xếp đơn điệu; tính đúng của thuật toán này: với mỗi `x`, các nút nhỏ hơn `x` trên ngăn xếp đều nằm ở nửa trái của `x` trong `Construct` (chúng đứng trước `x` và nhỏ hơn `x`), nút lớn nhất trong số đó là gốc của cây con trái; còn nút lớn hơn `x` gần nhất bên trái là tổ tiên mà `x` nằm ở nhánh phải của nó. Đây đúng là định nghĩa `Construct`, và ta đối chiếu bằng máy ngay bên dưới.

**Kiểm chứng bằng máy.** Oracle (Cách 1) dựng lại theo đúng định nghĩa `Construct(inorder + [val])`, độc lập với việc phân tích nhánh phải. Ta kiểm bốn cách trên rất nhiều cây ngẫu nhiên có seed (cây cực đại sinh từ hoán vị), so **chính xác cấu trúc** (`tree_to_list`) và kiểm tính chất (là cây cực đại, `inorder` mới = `inorder` cũ + `[val]`).

In [17]:
def random_max_tree_values(rng: random.Random, n: int, hi: int = 100) -> List[int]:
    return rng.sample(range(1, hi + 1), n)

rng = random.Random(1998)
for _ in range(400):
    n = rng.randint(1, 12)
    a = random_max_tree_values(rng, n, 30)
    val = rng.choice([x for x in range(1, 31) if x not in a])
    root_list = tree_to_list(construct(a))
    expected = tree_to_list(construct(a + [val]))
    for name, fn in (("rebuild", insert_rebuild), ("recursive", insert_recursive), ("iterative", insert_iterative),
                     ("stack", insert_stack), ("sol", sol.insertIntoMaxTree)):
        t = fn(build_tree(root_list), val)
        assert tree_to_list(t) == expected, (name, a, val)
        assert is_max_tree(t) and inorder(t) == a + [val]
print("400 cây ngẫu nhiên: 5 cách khớp Construct(a + [val]) về cấu trúc")

400 cây ngẫu nhiên: 5 cách khớp Construct(a + [val]) về cấu trúc


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Dựng lại bằng định nghĩa | `O(n^2)` tồi nhất | `O(n)` |
| 2. Đệ quy nhánh phải | `O(h)` | `O(h)` |
| 3. Vòng lặp nhánh phải | `O(h)` | `O(1)` |
| 4. Inorder + ngăn xếp đơn điệu | `O(n)` | `O(n)` |

`h` là độ dài nhánh phải ngoài cùng (`<= n`). Đo ở cỡ tối đa `n = 100` với ba dạng cây: nhánh phải rất dài (mảng giảm dần: mọi nút nối bằng con phải), nhánh phải rất ngắn (mảng tăng dần) và ngẫu nhiên:

In [18]:
rng = random.Random(11)
cases = {
    "nhánh phải dài (a giảm dần)": list(range(99, 0, -1)),
    "nhánh phải ngắn (a tăng dần)": list(range(1, 100)),
    "ngẫu nhiên": rng.sample(range(1, 100), 99),
}
for label, a in cases.items():
    base = tree_to_list(construct(a))
    val = 100
    expected = tree_to_list(construct(a + [val]))
    for name, fn in (("rebuild", insert_rebuild), ("Cách 2", sol_recursive), ("Cách 3", sol_iterative), ("Cách 4", sol_stack)):
        t0 = time.perf_counter()
        for _ in range(200):
            got = fn(build_tree(base), val)
        dt = (time.perf_counter() - t0) / 200 * 1000
        assert tree_to_list(got) == expected
        print(f"{label:30} {name:8}: {dt:7.3f} ms (gồm cả thời gian dựng cây từ danh sách)")

nhánh phải dài (a giảm dần)    rebuild :   0.147 ms (gồm cả thời gian dựng cây từ danh sách)


nhánh phải dài (a giảm dần)    Cách 2  :   0.028 ms (gồm cả thời gian dựng cây từ danh sách)
nhánh phải dài (a giảm dần)    Cách 3  :   0.026 ms (gồm cả thời gian dựng cây từ danh sách)
nhánh phải dài (a giảm dần)    Cách 4  :   0.073 ms (gồm cả thời gian dựng cây từ danh sách)


nhánh phải ngắn (a tăng dần)   rebuild :   0.186 ms (gồm cả thời gian dựng cây từ danh sách)


nhánh phải ngắn (a tăng dần)   Cách 2  :   0.031 ms (gồm cả thời gian dựng cây từ danh sách)
nhánh phải ngắn (a tăng dần)   Cách 3  :   0.028 ms (gồm cả thời gian dựng cây từ danh sách)
nhánh phải ngắn (a tăng dần)   Cách 4  :   0.058 ms (gồm cả thời gian dựng cây từ danh sách)


ngẫu nhiên                     rebuild :   0.093 ms (gồm cả thời gian dựng cây từ danh sách)
ngẫu nhiên                     Cách 2  :   0.025 ms (gồm cả thời gian dựng cây từ danh sách)


ngẫu nhiên                     Cách 3  :   0.029 ms (gồm cả thời gian dựng cây từ danh sách)


ngẫu nhiên                     Cách 4  :   0.072 ms (gồm cả thời gian dựng cây từ danh sách)


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất. Kết quả được so **chính xác cấu trúc** bằng `tree_to_list`.

In [19]:
def all_solutions():
    return {"rebuild": insert_rebuild, "recursive": insert_recursive, "iterative": insert_iterative, "stack": insert_stack,
            "sol_recursive": sol_recursive, "sol_iterative": sol_iterative, "sol_stack": sol_stack, "final": sol.insertIntoMaxTree}

def check_all(root_list, val, expected=None):
    a = inorder(build_tree(root_list))
    ref = tree_to_list(construct(a + [val]))
    for name, fn in all_solutions().items():
        t = fn(build_tree(root_list), val)
        assert tree_to_list(t) == ref, (name, root_list, val, tree_to_list(t), ref)
        assert is_max_tree(t) and inorder(t) == a + [val]
    if expected is not None:
        assert ref == expected, (root_list, val, ref, expected)
    return ref

# 12.1 ví dụ đề
for vals, val, expected in examples:
    check_all(vals, val, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1], 2, [2, 1]),                               # val lớn hơn gốc duy nhất: gốc mới, cây cũ là con trái
    ([2], 1, [2, None, 1]),                         # val nhỏ hơn: thành con phải
    ([3, 1, 2], 4, [4, 3, None, 1, 2]),             # a = [1,3,2]; val lớn nhất: bọc cả cây làm con trái
    ([3, 1, 2], 5, [5, 3, None, 1, 2]),             # tương tự, val = 5
    ([4, 1, 3, None, None, 2], 6, [6, 4, None, 1, 3, None, None, 2]),
    ([5, 3, None, 2], 4, [5, 3, 4, 2]),             # a = [2,3,5]; 4 < 5 nên thành con phải của 5 (đang rỗng)
    ([5, 2, 4, None, 1], 3, [5, 2, 4, None, 1, None, 3]),         # ví dụ 2: chèn dưới nhánh phải
    ([10, 4, 9, None, 3, 8], 7, None),              # a = [4,3,10,8,9]: 7 < 9 nên xuống tới cuối nhánh phải; kết quả do oracle xác nhận
]
for vals, val, expected in edge:
    check_all(vals, val, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 8 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed so với oracle Construct(a + [val]))
def wrong_lose_left(root, val):                   # nút mới nhưng quên gắn cây cũ làm con trái
    if root is None or val > root.val:
        return TreeNode(val)
    root.right = wrong_lose_left(root.right, val)
    return root

def wrong_go_left(root, val):                     # đi xuống nhánh TRÁI thay vì phải
    if root is None or val > root.val:
        return TreeNode(val, left=root)
    root.left = wrong_go_left(root.left, val)
    return root

def wrong_append_last(root, val):                 # gắn val làm con phải của nút cuối nhánh phải, bất kể giá trị
    if root is None:
        return TreeNode(val)
    cur = root
    while cur.right is not None:
        cur = cur.right
    cur.right = TreeNode(val)
    return root

def wrong_overwrite_root(root, val):              # ghi đè giá trị gốc khi val lớn hơn
    if root is None:
        return TreeNode(val)
    if val > root.val:
        root.val = val
        return root
    root.right = wrong_overwrite_root(root.right, val)
    return root

def wrong_return_old_root(root, val):             # vòng lặp nhưng quên trả gốc mới khi val lớn nhất
    parent, cur = None, root
    while cur is not None and cur.val > val:
        parent, cur = cur, cur.right
    node = TreeNode(val, left=cur)
    if parent is not None:
        parent.right = node
    return root

def find_counter(wrong, seed=9, tries=20000):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 6)
        a = rng.sample(range(1, 11), n)
        val = rng.choice([x for x in range(1, 11) if x not in a])
        root_list = tree_to_list(construct(a))
        try:
            got = tree_to_list(wrong(build_tree(root_list), val))
        except Exception:
            return root_list, val
        if got != tree_to_list(construct(a + [val])):
            return root_list, val
    return None

counters = {}
for name, fn in (("mất cây cũ (không gắn con trái)", wrong_lose_left), ("đi xuống nhánh trái", wrong_go_left),
                 ("gắn vào cuối nhánh phải", wrong_append_last), ("ghi đè giá trị gốc", wrong_overwrite_root),
                 ("quên trả gốc mới", wrong_return_old_root)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    root_list, val = case
    print(f"{name:32} root={root_list} val={val}: sai -> {tree_to_list(fn(build_tree(root_list), val))}, đúng -> {check_all(root_list, val)}")
print("12.3 phản ví dụ: OK")

mất cây cũ (không gắn con trái)  root=[10, None, 6, None, 5, None, 2] val=3: sai -> [10, None, 6, None, 5, None, 3], đúng -> [10, None, 6, None, 5, None, 3, 2]
đi xuống nhánh trái              root=[10, None, 6, None, 5, None, 2] val=3: sai -> [10, 3, 6, None, None, None, 5, None, 2], đúng -> [10, None, 6, None, 5, None, 3, 2]
gắn vào cuối nhánh phải          root=[10, None, 6, None, 5, None, 2] val=3: sai -> [10, None, 6, None, 5, None, 2, None, 3], đúng -> [10, None, 6, None, 5, None, 3, 2]
ghi đè giá trị gốc               root=[10, None, 6, None, 5, None, 2] val=3: sai -> [10, None, 6, None, 5, None, 3], đúng -> [10, None, 6, None, 5, None, 3, 2]
quên trả gốc mới                 root=[2, 1] val=5: sai -> [2, 1], đúng -> [5, 2, None, 1]
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên: cây từ hoán vị, n tới 99, val ngẫu nhiên chưa có trong cây
rng = random.Random(4998)
for _ in range(800):
    n = rng.randint(1, 99)
    a = rng.sample(range(1, 101), n)
    val = rng.choice([x for x in range(1, 101) if x not in a])
    check_all(tree_to_list(construct(a)), val)
print("12.4 ngẫu nhiên (800 cây, n tới 99): OK")

# 12.5 dạng đặc biệt: mảng tăng dần, giảm dần, núi, thung lũng; val nhỏ nhất, lớn nhất, ở giữa
specials = [list(range(1, 100)), list(range(99, 0, -1)), list(range(1, 50)) + list(range(99, 49, -1)),
            list(range(50, 0, -1)) + list(range(51, 100))]
for a in specials:
    a = a[:99]
    for val in (100, 1 if 1 not in a else 100, 50 if 50 not in a else 100):
        if val in a:
            continue
        check_all(tree_to_list(construct(a)), val)
print("12.5 dạng đặc biệt: OK")

# 12.6 tính chất: chèn lần lượt từng phần tử vào cây rỗng cho đúng Construct(a); số nút tăng 1; nhánh phải ngoài cùng chứa val hoặc gốc mới
rng = random.Random(6998)
for _ in range(300):
    n = rng.randint(1, 40)
    a = rng.sample(range(1, 101), n)
    t = None
    for x in a:
        t = sol.insertIntoMaxTree(t, x)
    assert tree_to_list(t) == tree_to_list(construct(a))
    val = rng.choice([x for x in range(1, 101) if x not in a])
    before = inorder(t)
    t2 = sol.insertIntoMaxTree(t, val)
    assert len(inorder(t2)) == len(before) + 1 and val in right_spine(t2)
print("12.6 tính chất (chèn lần lượt = Construct; val luôn nằm trên nhánh phải ngoài cùng): OK")

12.4 ngẫu nhiên (800 cây, n tới 99): OK
12.5 dạng đặc biệt: OK
12.6 tính chất (chèn lần lượt = Construct; val luôn nằm trên nhánh phải ngoài cùng): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Mất cây cũ (không gắn làm con trái)

Tạo nút `val` mà quên đặt `left=root` làm mất toàn bộ phần cây cũ bị "đẩy xuống".

In [22]:
root_list, val = counters["mất cây cũ (không gắn con trái)"]
bad = tree_to_list(wrong_lose_left(build_tree(root_list), val))
print("root =", root_list, "val =", val, "| sai ->", bad, "| đúng ->", tree_to_list(Solution().insertIntoMaxTree(build_tree(root_list), val)))
assert bad != tree_to_list(Solution().insertIntoMaxTree(build_tree(root_list), val))

root = [10, None, 6, None, 5, None, 2] val = 3 | sai -> [10, None, 6, None, 5, None, 3] | đúng -> [10, None, 6, None, 5, None, 3, 2]


Vì sao sai: các nút nhỏ hơn `val` ở bên trái của vị trí chèn vẫn thuộc `b` (nằm **trước** `val` trong mảng), nên phải trở thành con trái của nút mới. **Đúng:** `TreeNode(val, left=root)`.

### Lỗi 2 — Đi xuống nhánh trái

`val` đứng **cuối** mảng nên luôn thuộc nhánh **phải**. Đi xuống trái đặt nó ở sai chỗ.

In [23]:
root_list, val = counters["đi xuống nhánh trái"]
bad = tree_to_list(wrong_go_left(build_tree(root_list), val))
print("root =", root_list, "val =", val, "| sai ->", bad, "| đúng ->", tree_to_list(Solution().insertIntoMaxTree(build_tree(root_list), val)))
assert bad != tree_to_list(Solution().insertIntoMaxTree(build_tree(root_list), val))

root = [10, None, 6, None, 5, None, 2] val = 3 | sai -> [10, 3, 6, None, None, None, 5, None, 2] | đúng -> [10, None, 6, None, 5, None, 3, 2]


Vì sao sai: nửa trái chứa các phần tử **đứng trước** gốc trong mảng; `val` đứng sau tất cả. **Đúng:** `root.right = insert(root.right, val)`.

### Lỗi 3 — Gắn `val` làm con phải của nút cuối nhánh phải bất kể giá trị

Khi `val` lớn hơn một số nút trên nhánh phải, nó phải **chen** vào giữa, không đứng cuối.

In [24]:
root_list, val = counters["gắn vào cuối nhánh phải"]
bad = tree_to_list(wrong_append_last(build_tree(root_list), val))
print("root =", root_list, "val =", val, "| sai ->", bad, "| đúng ->", tree_to_list(Solution().insertIntoMaxTree(build_tree(root_list), val)))
assert bad != tree_to_list(Solution().insertIntoMaxTree(build_tree(root_list), val))
assert not is_max_tree(wrong_append_last(build_tree(root_list), val)) or bad != tree_to_list(Solution().insertIntoMaxTree(build_tree(root_list), val))

root = [10, None, 6, None, 5, None, 2] val = 3 | sai -> [10, None, 6, None, 5, None, 2, None, 3] | đúng -> [10, None, 6, None, 5, None, 3, 2]


Vì sao sai: cây không còn là cây cực đại (hoặc sai cấu trúc) khi nút cha nhỏ hơn con. **Đúng:** dừng ở nút đầu tiên nhỏ hơn `val` và chèn trước nó.

### Lỗi 4 — Ghi đè giá trị gốc

Đổi `root.val = val` làm mất giá trị cũ và phá cấu trúc.

In [25]:
root_list, val = counters["ghi đè giá trị gốc"]
bad = tree_to_list(wrong_overwrite_root(build_tree(root_list), val))
print("root =", root_list, "val =", val, "| sai ->", bad, "| đúng ->", tree_to_list(Solution().insertIntoMaxTree(build_tree(root_list), val)))
assert bad != tree_to_list(Solution().insertIntoMaxTree(build_tree(root_list), val))

root = [10, None, 6, None, 5, None, 2] val = 3 | sai -> [10, None, 6, None, 5, None, 3] | đúng -> [10, None, 6, None, 5, None, 3, 2]


Vì sao sai: số nút phải tăng thêm một, và giá trị cũ vẫn phải có mặt. **Đúng:** tạo nút mới, nối cây cũ vào làm con trái.

### Lỗi 5 — Quên trả về gốc mới khi `val` lớn nhất

Trong cách lặp, nếu `val` lớn hơn gốc thì `parent` là `None`; hàm phải trả về **nút mới**, không phải `root` cũ.

In [26]:
root_list, val = counters["quên trả gốc mới"]
bad = tree_to_list(wrong_return_old_root(build_tree(root_list), val))
print("root =", root_list, "val =", val, "| sai ->", bad, "| đúng ->", tree_to_list(Solution().insertIntoMaxTree(build_tree(root_list), val)))
assert bad != tree_to_list(Solution().insertIntoMaxTree(build_tree(root_list), val))

root = [2, 1] val = 5 | sai -> [2, 1] | đúng -> [5, 2, None, 1]


Vì sao sai: nút mới (gốc của cây kết quả) bị bỏ rơi, trả về cây cũ không chứa `val`. **Đúng:** `if parent is None: return node`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "cây cực đại / Cartesian tree + thêm phần tử vào cuối mảng" ⇒ chỉ **nhánh phải ngoài cùng** thay đổi; chèn tại vị trí đầu tiên có nút nhỏ hơn `val`.
- **Inorder của cây cực đại là mảng gốc `a`**; luôn nghĩ "cây ↔ mảng" khi quên cấu trúc.
- Mẫu "chèn vào cây": `if node is None or điều_kiện: return Node(..., left=node)`, ngược lại đệ quy xuống một con rồi gán `node.child = ...; return node`.
- Dựng cây cực đại từ mảng trong `O(n)` bằng **ngăn xếp giảm dần** (bài tập 5).
- Cùng họ: Maximum Binary Tree (654), Insert into a Binary Search Tree (701), Cartesian Tree.
- Test nên có: `val` lớn hơn gốc, `val` nhỏ nhất, `val` xen giữa nhánh phải, cây một nút, nhánh phải rất dài.

## 15. Biến thể

1. **Chèn `val` vào vị trí bất kỳ** của `a` (không chỉ cuối): cần tách cây theo vị trí (split) hoặc dựng lại bằng ngăn xếp `O(n)`.
2. **Xóa phần tử cuối** của `a`: ngược lại với bài này (bỏ nút cuối trên nhánh phải và nối cây con trái lên).
3. **Chèn lần lượt từng phần tử** vào cây rỗng cho đúng `Construct(a)` (đã kiểm ở mục 12.6).
4. Bài liên quan: 654, 701.

Dưới đây là biến thể 1 (chèn `val` vào vị trí `pos` của `a`), kiểm bằng oracle `construct`:

In [27]:
def insert_at(root: Optional[TreeNode], val: int, pos: int) -> Optional[TreeNode]:
    a = inorder(root)
    a.insert(pos, val)
    return construct_stack(a)

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(0, 12)
    a = rng.sample(range(1, 40), n)
    val = rng.choice([x for x in range(1, 40) if x not in a])
    pos = rng.randint(0, n)
    got = tree_to_list(insert_at(construct(a), val, pos))
    assert got == tree_to_list(construct(a[:pos] + [val] + a[pos:]))
    if pos == n:                                           # chèn ở cuối: trùng với bài này
        assert got == tree_to_list(Solution().insertIntoMaxTree(construct(a), val))
print("biến thể chèn ở vị trí bất kỳ: khớp Construct; khi pos = n trùng với bài 998")

biến thể chèn ở vị trí bất kỳ: khớp Construct; khi pos = n trùng với bài 998


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng lại bằng định nghĩa | inorder + `Construct(b)` | `O(n^2)` | `O(n)` | Oracle |
| 2. Đệ quy nhánh phải | Bọc cây cũ làm con trái | `O(h)` | `O(h)` | **Cách nộp chính** |
| 3. Vòng lặp nhánh phải | Giữ nút cha | `O(h)` | `O(1)` | Không đệ quy |
| 4. Inorder + ngăn xếp | Cartesian tree `O(n)` | `O(n)` | `O(n)` | Khi cần dựng lại hoàn toàn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
t = build_tree([4, 1, 3, None, None, 2])
print(inorder(t), right_spine(t))
print(tree_to_list(construct([3, 1, 2])))
print(is_max_tree(build_tree([3, 1, 2])), is_max_tree(build_tree([1, 3])))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # hai danh sách (inorder, nhánh phải)
my_guess_2 = None      # danh sách theo mức
my_guess_3 = None      # hai bool

### Bài 2 — Tính tay (dễ-vừa)

`root = [6, 3, 4, 2]` (gốc `6`, con trái `3` có con trái `2`, con phải `4`), `val = 5`. Hãy xác định `a`, `b` rồi ghi danh sách theo mức của kết quả (dùng `None` cho nút trống, bỏ các `None` thừa ở cuối).

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành đệ quy.

In [30]:
def exercise_3(root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
    # TODO 1: nếu root rỗng hoặc val > root.val: trả về nút mới có con trái là root
    # TODO 2: ngược lại root.right = exercise_3(root.right, val)
    # TODO 3: trả về root
    pass


# for vals, val, expected in examples:
#     assert tree_to_list(exercise_3(build_tree(vals), val)) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, tự viết `exercise_4(root, val)` bằng **vòng lặp** (giữ `parent` và `cur`).

In [31]:
def exercise_4(root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
    # TODO
    pass


# for vals, val, expected in examples:
#     assert tree_to_list(exercise_4(build_tree(vals), val)) == expected

### Bài 5 — Mini challenge: dựng cây cực đại từ mảng (khó)

Viết `exercise_5(nums)` trả về gốc của cây cực đại dựng từ `nums` (các phần tử khác nhau) theo `Construct` (bài 654), trong `O(n)` bằng ngăn xếp.

```python
tree_to_list(exercise_5([3, 2, 1, 6, 0, 5])) -> [6, 3, 5, None, 2, 0, None, None, 1]
tree_to_list(exercise_5([3, 2, 1]))           -> [3, None, 2, None, 1]
tree_to_list(exercise_5([]))                  -> []
```
Gợi ý: ngăn xếp giảm dần; nút bị pop cuối cùng là con trái của nút mới; nút mới là con phải của nút trên cùng.

In [32]:
def exercise_5(nums: List[int]) -> Optional[TreeNode]:
    # TODO
    pass


# assert tree_to_list(exercise_5([3, 2, 1, 6, 0, 5])) == [6, 3, 5, None, 2, 0, None, None, 1]
# assert tree_to_list(exercise_5([3, 2, 1])) == [3, None, 2, None, 1]
# assert tree_to_list(exercise_5([])) == []

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: inorder trả về mảng `a`; nhánh phải ngoài cùng đi từ gốc rồi luôn sang phải.
- Mức 2: `[4,1,3,null,null,2]` có inorder `[1, 4, 2, 3]`; nhánh phải: `4 -> 3`.
- Mức 3: dòng 1 `[1, 4, 2, 3] [4, 3]`; dòng 2: `Construct([3,1,2])` có gốc `3`, con trái rỗng, con phải là `2` (có con trái `1`): `[3, None, 2, 1]`; dòng 3: `True False`.

**Bài 2**
- Mức 1: `a = inorder([6,3,4,2]) = [2, 3, 6, 4]`, `b = [2, 3, 6, 4, 5]`.
- Mức 2: gốc vẫn `6`; `5 < 6` nên đi xuống con phải `4`; `5 > 4` nên `5` thay vào, `4` thành con trái của `5`.
- Mức 3: `[6, 3, 5, 2, None, 4]`.

**Bài 3**
- Mức 1: `if root is None or val > root.val: return TreeNode(val, left=root)`.
- Mức 2: `root.right = exercise_3(root.right, val)`.
- Mức 3: `return root`.

**Bài 4**
- Mức 1: `parent, cur = None, root`; `while cur and cur.val > val: parent, cur = cur, cur.right`.
- Mức 2: `node = TreeNode(val, left=cur)`.
- Mức 3: `if parent is None: return node`; `parent.right = node`; `return root`.

**Bài 5**
- Mức 1: duyệt từng `x`; `node = TreeNode(x)`; `last = None`.
- Mức 2: `while stack and stack[-1].val < x: last = stack.pop()`; `node.left = last`.
- Mức 3: `if stack: stack[-1].right = node`; `stack.append(node)`; trả về `stack[0]` hoặc `None`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
t = build_tree([4, 1, 3, None, None, 2])
print(inorder(t), right_spine(t))
print(tree_to_list(construct([3, 1, 2])))
print(is_max_tree(build_tree([3, 1, 2])), is_max_tree(build_tree([1, 3])))

assert (inorder(t), right_spine(t)) == ([1, 4, 2, 3], [4, 3])
assert tree_to_list(construct([3, 1, 2])) == [3, None, 2, 1]
assert (is_max_tree(build_tree([3, 1, 2])), is_max_tree(build_tree([1, 3]))) == (True, False)

[1, 4, 2, 3] [4, 3]
[3, None, 2, 1]
True False


### Đáp án Bài 2

`a = [2, 3, 6, 4]` (inorder), `b = [2, 3, 6, 4, 5]`. `Construct(b)`: gốc `6`; trái `Construct([2, 3])` = nút `3` có con trái `2`; phải `Construct([4, 5])` = nút `5` có con trái `4`. Danh sách theo mức: `[6, 3, 5, 2, None, 4]`.

In [34]:
hand_answer_solution = [6, 3, 5, 2, None, 4]
t = build_tree([6, 3, 4, 2])
assert inorder(t) == [2, 3, 6, 4]
assert tree_to_list(Solution().insertIntoMaxTree(t, 5)) == hand_answer_solution
assert tree_to_list(construct([2, 3, 6, 4, 5])) == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
    if root is None or val > root.val:
        return TreeNode(val, left=root)
    root.right = exercise_3_solution(root.right, val)
    return root

for vals, val, expected in examples:
    assert tree_to_list(exercise_3_solution(build_tree(vals), val)) == expected
rng = random.Random(3)
for _ in range(200):
    a = rng.sample(range(1, 60), rng.randint(1, 15))
    val = rng.choice([x for x in range(1, 60) if x not in a])
    assert tree_to_list(exercise_3_solution(construct(a), val)) == tree_to_list(construct(a + [val]))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
    parent, cur = None, root
    while cur is not None and cur.val > val:
        parent, cur = cur, cur.right
    node = TreeNode(val, left=cur)
    if parent is None:
        return node
    parent.right = node
    return root

for vals, val, expected in examples:
    assert tree_to_list(exercise_4_solution(build_tree(vals), val)) == expected
rng = random.Random(4)
for _ in range(200):
    a = rng.sample(range(1, 60), rng.randint(1, 15))
    val = rng.choice([x for x in range(1, 60) if x not in a])
    assert tree_to_list(exercise_4_solution(construct(a), val)) == tree_to_list(construct(a + [val]))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(nums: List[int]) -> Optional[TreeNode]:
    stack: List[TreeNode] = []
    for x in nums:
        node = TreeNode(x)
        last = None
        while stack and stack[-1].val < x:
            last = stack.pop()
        node.left = last
        if stack:
            stack[-1].right = node
        stack.append(node)
    return stack[0] if stack else None

assert tree_to_list(exercise_5_solution([3, 2, 1, 6, 0, 5])) == [6, 3, 5, None, 2, 0, None, None, 1]
assert tree_to_list(exercise_5_solution([3, 2, 1])) == [3, None, 2, None, 1]
assert tree_to_list(exercise_5_solution([])) == []
rng = random.Random(5)
for _ in range(300):
    a = rng.sample(range(1, 100), rng.randint(0, 30))
    assert tree_to_list(exercise_5_solution(a)) == tree_to_list(construct(a))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (đệ quy theo nhánh phải)

```python
class Solution:
    def insertIntoMaxTree(self, root: Optional[TreeNode], val: int) -> Optional[TreeNode]:
        if root is None or val > root.val:
            return TreeNode(val, left=root)          # nút mới: cây cũ thành con trái
        root.right = self.insertIntoMaxTree(root.right, val)
        return root
```

Công thức nhớ nhanh: `val` ở cuối mảng ⇒ luôn thuộc nhánh phải; dừng ở nút đầu tiên nhỏ hơn `val` (hoặc rỗng), chèn nút mới và đặt cây vừa bỏ qua làm con trái.

### Checklist trước khi submit

- [ ] Nút mới có `left=root` (không làm mất cây cũ)?
- [ ] Đi xuống **nhánh phải** (không phải trái)?
- [ ] Điều kiện dừng: `root is None or val > root.val`?
- [ ] Gán lại `root.right = ...` và **trả về `root`** ở nhánh đệ quy?
- [ ] Khi `val` lớn nhất, trả về **nút mới** làm gốc (đệ quy tự xử lý; cách lặp cần `if parent is None`)?
- [ ] Đã thử: `val` lớn nhất, nhỏ nhất, xen giữa nhánh phải, cây một nút, ví dụ 1-3, kết quả so bằng `tree_to_list`?